# INJECTION-R2: Joint Fine-tuning (MiniLM Encoder Unfrozen)

**Stage:** R-2 of the INJECTION pipeline  
**Input:** `r1_best_weights.h5` (heads-only warm-up from R1)  
**Output:** `r2_best_weights.h5` (end-to-end fine-tuned)  

### What this stage does
- Unfreezes the MiniLM-L6-v2 encoder for end-to-end training
- Uses **differential learning rates**: encoder `2e-5`, heads `5e-4`
- Alternating batch strategy: ATS+Domain batches interleaved with RSG batches
- Loss weights: ATS=0.35, Domain=0.35, RSG=0.30 (canonical)

### Target Gates
| Metric | Target |
|--------|--------|
| ATS val MAE (0–100) | < 8.0 |
| ATS test MAE (0–100) | < 8.5 |
| Domain val F1 (macro) | > 0.82 |
| RSG val Accuracy | > 0.55 |

### Required files on Google Drive
Upload all 4 files **directly into** your `ATS_R2/` folder (no subfolders needed):
```
MyDrive/ATS_R2/
  ats_tokenized.npz        ← from data/tokenized/
  rsg_tokenized.npz        ← from data/tokenized/
  r1_best_weights.h5       ← from model/unified_model/
  data_splits.json         ← from model/unified_model/
```
Outputs (`r2_best_weights.h5`, `r2_training_log.csv`) will be auto-saved to `ATS_R2/model/unified_model/`.

> **Runtime:** Set to **T4 GPU** via `Runtime → Change runtime type → T4 GPU`

In [ ]:
# ── Cell 1: GPU check ────────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
if result.returncode != 0:
    print('⚠️  No GPU detected!')
    print('Go to Runtime → Change runtime type → Hardware accelerator → T4 GPU')
    print('Then re-run all cells.')
else:
    print(result.stdout)

In [ ]:
# ── Cell 2: Install dependencies ────────────────────────────────────────────
!pip install -q "transformers>=4.35.0" scikit-learn
print('Dependencies ready.')

In [ ]:
# ── Cell 3: Mount Google Drive & configure paths ─────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

# ╔══════════════════════════════════════════════════════════╗
# ║  CONFIGURE: set this to your ATS_R2 folder on Drive     ║
# ╚══════════════════════════════════════════════════════════╝
DRIVE_ROOT = '/content/drive/MyDrive/ATS_R2'

import os
from pathlib import Path

# Input files — uploaded directly to ATS_R2/ root
DATA_DIR        = Path(DRIVE_ROOT)                              # .npz files live here
R1_WEIGHTS      = str(Path(DRIVE_ROOT) / 'r1_best_weights.h5')
SPLITS_JSON     = str(Path(DRIVE_ROOT) / 'data_splits.json')

# Output files — saved into model/unified_model/ subfolder
UNIFIED_DIR     = Path(DRIVE_ROOT) / 'model' / 'unified_model'
UNIFIED_DIR.mkdir(parents=True, exist_ok=True)
R2_BEST_PATH    = str(UNIFIED_DIR / 'r2_best_weights.h5')
R2_CKPT_PATH    = str(UNIFIED_DIR / 'r2_best_in_progress.weights.h5')
R2_LOG_PATH     = str(UNIFIED_DIR / 'r2_training_log.csv')

print(f'Drive root : {DRIVE_ROOT}')
print(f'Data dir   : {DATA_DIR}')
print(f'R1 weights : {R1_WEIGHTS}')
print(f'Splits JSON: {SPLITS_JSON}')
print(f'Outputs  → : {UNIFIED_DIR}')

In [ ]:
# ── Cell 4: Verify required files ────────────────────────────────────────────
required = [
    str(DATA_DIR / 'ats_tokenized.npz'),
    str(DATA_DIR / 'rsg_tokenized.npz'),
    R1_WEIGHTS,
    SPLITS_JSON,
]
all_ok = True
for f in required:
    exists = os.path.exists(f)
    size   = f'{os.path.getsize(f) / 1e6:.1f} MB' if exists else 'MISSING'
    status = '✅' if exists else '❌'
    print(f'  {status}  {size:>10}   {f}')
    all_ok = all_ok and exists

if not all_ok:
    raise FileNotFoundError('One or more required files are missing. Upload them to Drive first.')
print('\nAll required files found — ready to train.')

In [ ]:
# ── Cell 5: Imports & environment ────────────────────────────────────────────
import os
os.environ['TF_USE_LEGACY_KERAS'] = '1'
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'

import csv, json, math, sys
import numpy as np
import tensorflow as tf
import logging
logging.getLogger('tensorflow').setLevel(logging.ERROR)
tf.get_logger().setLevel('ERROR')
from sklearn.metrics import f1_score

print(f'TensorFlow : {tf.__version__}')
print(f'GPU devices: {tf.config.list_physical_devices("GPU")}')

In [ ]:
# ── Cell 6: Hyperparameters ───────────────────────────────────────────────────
MINILM_MODEL_NAME = 'sentence-transformers/all-MiniLM-L6-v2'
SEQ_LEN           = 128
EMB_DIM           = 384

ENC_LR      = 2e-5
HEAD_LR     = 5e-4
BATCH_SIZE  = 32
MAX_EPOCHS  = 50
PATIENCE    = 10
W_ATS       = 0.35
W_DOM       = 0.35
W_RSG       = 0.30
DOM_CLS_W   = [1.4, 0.8, 0.9, 1.0, 1.5, 0.9, 1.0]  # per domain 0–6

MAE_REGRESSION_LIMIT = 3.0   # hard-stop: val MAE must not exceed R1 + 3.0

GATE_VAL_MAE  = 8.0
GATE_TEST_MAE = 8.5
GATE_DOM_F1   = 0.82
GATE_RSG_ACC  = 0.55
DOMAIN_NAMES  = ['IT', 'Non-IT', 'Design', 'Healthcare', 'Finance', 'Legal', 'Edu']

print('Hyperparameters configured.')
print(f'  ENC_LR={ENC_LR}  HEAD_LR={HEAD_LR}  BATCH={BATCH_SIZE}  MAX_EPOCHS={MAX_EPOCHS}  PATIENCE={PATIENCE}')
print(f'  Loss weights: ATS={W_ATS}  DOM={W_DOM}  RSG={W_RSG}')

In [ ]:
# ── Cell 7: MiniLM model definition (self-contained, no local imports) ────────
from transformers import TFAutoModel

def mean_pool_l2(last_hidden, attention_mask):
    """Attention-mask weighted mean pool + L2 normalise → [B, 384]."""
    mask    = tf.cast(tf.expand_dims(attention_mask, -1), tf.float32)
    sum_emb = tf.reduce_sum(last_hidden * mask, axis=1)
    count   = tf.maximum(tf.reduce_sum(mask, axis=1), 1e-9)
    return tf.nn.l2_normalize(sum_emb / count, axis=-1)


class MiniLMEncoderLayer(tf.keras.layers.Layer):
    """Wraps HuggingFace TFBertModel as a Keras layer."""
    def __init__(self, bert_model, **kwargs):
        kwargs.setdefault('trainable', False)
        super().__init__(**kwargs)
        self._bert = bert_model
        self._bert.trainable = False

    def call(self, inputs, training=False):
        input_ids, attention_mask = inputs
        out = self._bert(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=tf.zeros_like(input_ids),
            training=False,
        )
        return mean_pool_l2(out.last_hidden_state, attention_mask)

    def get_config(self):
        return super().get_config()


def build_unified_minilm_model(bert_model):
    """Unified ATS + Domain + RSG model with frozen MiniLM encoder."""
    resume_ids  = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='resume_input_ids')
    resume_mask = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='resume_attention_mask')
    jd_ids      = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='jd_input_ids')
    jd_mask     = tf.keras.Input((SEQ_LEN,), dtype=tf.int32, name='jd_attention_mask')

    encoder    = MiniLMEncoderLayer(bert_model, name='minilm_encoder')
    resume_emb = encoder([resume_ids,  resume_mask])
    jd_emb     = encoder([jd_ids,      jd_mask])

    cosine_sim = tf.keras.layers.Dot(axes=1, normalize=True,  name='cosine_sim')([resume_emb, jd_emb])
    dot_prod   = tf.keras.layers.Dot(axes=1, normalize=False, name='dot_prod')  ([resume_emb, jd_emb])
    ats_feat   = tf.keras.layers.Concatenate(name='ats_features')([resume_emb, jd_emb, cosine_sim, dot_prod])

    # HEAD 1 — ATS Score (770 → 256 → 64 → 1)
    x1        = tf.keras.layers.Dense(256, activation='relu',    name='ats_dense1')(ats_feat)
    x1        = tf.keras.layers.Dropout(0.3,                     name='ats_drop1')(x1)
    x1        = tf.keras.layers.Dense(64,  activation='relu',    name='ats_dense2')(x1)
    x1        = tf.keras.layers.Dropout(0.2,                     name='ats_drop2')(x1)
    ats_score = tf.keras.layers.Dense(1, activation='sigmoid',   name='ats_score')(x1)

    # HEAD 2 — Domain (384 → 256 → 128 → 7)
    x2           = tf.keras.layers.Dense(256, activation='relu', name='dom_dense1')(jd_emb)
    x2           = tf.keras.layers.Dropout(0.3,                  name='dom_drop1')(x2)
    x2           = tf.keras.layers.Dense(128, activation='relu', name='dom_dense2')(x2)
    x2           = tf.keras.layers.Dropout(0.2,                  name='dom_drop2')(x2)
    domain_probs = tf.keras.layers.Dense(7, activation='softmax',name='domain_probs')(x2)

    # HEAD 3 — RSG (384 → 512→BN → 256→BN → 128→BN → 46)
    x3           = tf.keras.layers.Dense(512, activation='relu', name='rsg_dense1')(resume_emb)
    x3           = tf.keras.layers.BatchNormalization(           name='rsg_bn1')(x3)
    x3           = tf.keras.layers.Dropout(0.4,                  name='rsg_drop1')(x3)
    x3           = tf.keras.layers.Dense(256, activation='relu', name='rsg_dense2')(x3)
    x3           = tf.keras.layers.BatchNormalization(           name='rsg_bn2')(x3)
    x3           = tf.keras.layers.Dropout(0.3,                  name='rsg_drop2')(x3)
    x3           = tf.keras.layers.Dense(128, activation='relu', name='rsg_dense3')(x3)
    x3           = tf.keras.layers.BatchNormalization(           name='rsg_bn3')(x3)
    x3           = tf.keras.layers.Dropout(0.3,                  name='rsg_drop3')(x3)
    rsg_template = tf.keras.layers.Dense(46, activation='softmax',name='rsg_template')(x3)

    return tf.keras.Model(
        inputs=[resume_ids, resume_mask, jd_ids, jd_mask],
        outputs=[ats_score, domain_probs, rsg_template],
        name='unified_ats_minilm',
    )


print('Model definition ready.')

In [ ]:
# ── Cell 8: Load tokenized data & splits ─────────────────────────────────────
print('Loading tokenized data...')
ats_data = np.load(str(DATA_DIR / 'ats_tokenized.npz'))
rsg_data = np.load(str(DATA_DIR / 'rsg_tokenized.npz'))

with open(SPLITS_JSON) as f:
    splits = json.load(f)

ats_tr_idx = np.array(splits['ats_train'])
ats_vl_idx = np.array(splits['ats_val'])
ats_ts_idx = np.array(splits['ats_test'])
rsg_tr_idx = np.array(splits['rsg_train'])
rsg_vl_idx = np.array(splits['rsg_val'])

ATS_KEYS = (
    'resume_input_ids', 'resume_attention_mask',
    'jd_input_ids', 'jd_attention_mask',
    'ats_scores', 'domain_labels',
)
RSG_KEYS = ('profile_input_ids', 'profile_attention_mask', 'rsg_labels')

ats_tr = {k: ats_data[k][ats_tr_idx] for k in ATS_KEYS}
ats_vl = {k: ats_data[k][ats_vl_idx] for k in ATS_KEYS}
ats_ts = {k: ats_data[k][ats_ts_idx] for k in ATS_KEYS}
rsg_tr = {k: rsg_data[k][rsg_tr_idx] for k in RSG_KEYS}
rsg_vl = {k: rsg_data[k][rsg_vl_idx] for k in RSG_KEYS}

n_ats_tr = len(ats_tr_idx)
n_rsg_tr = len(rsg_tr_idx)
print(f'  ATS  train={n_ats_tr:,}  val={len(ats_vl_idx):,}  test={len(ats_ts_idx):,}')
print(f'  RSG  train={n_rsg_tr:,}  val={len(rsg_vl_idx):,}')

In [ ]:
# ── Cell 9: Build model & load R1 weights ────────────────────────────────────
print(f'Loading MiniLM encoder from HuggingFace ({MINILM_MODEL_NAME})...')
bert_model = TFAutoModel.from_pretrained(MINILM_MODEL_NAME, from_pt=True)
bert_model.trainable = False

model = build_unified_minilm_model(bert_model)
model.load_weights(R1_WEIGHTS)
print(f'R1 weights loaded: {R1_WEIGHTS}')
model.summary(line_length=80)

In [ ]:
# ── Cell 10: Loss helpers & evaluation functions ──────────────────────────────
dom_w_tf = tf.constant(DOM_CLS_W, dtype=tf.float32)
_sce     = tf.keras.losses.SparseCategoricalCrossentropy(from_logits=False)


def _ats_mae(y_true, y_pred):
    return tf.reduce_mean(tf.abs(tf.cast(y_true, tf.float32) - tf.squeeze(y_pred, axis=-1)))


def _dom_loss(y_true, y_pred):
    ce = tf.keras.losses.sparse_categorical_crossentropy(y_true, y_pred)
    w  = tf.gather(dom_w_tf, tf.cast(y_true, tf.int32))
    return tf.reduce_mean(ce * w)


def _rsg_loss(y_true, y_pred):
    return _sce(y_true, y_pred)


def eval_ats(data):
    """Returns (mae_100, dom_f1, dom_ce, dom_preds_array)."""
    n = len(data['ats_scores'])
    ats_preds, dom_preds = [], []
    for s in range(0, n, BATCH_SIZE):
        e = min(s + BATCH_SIZE, n)
        ap, dp, _ = model(
            [data['resume_input_ids'][s:e], data['resume_attention_mask'][s:e],
             data['jd_input_ids'][s:e],     data['jd_attention_mask'][s:e]],
            training=False,
        )
        ats_preds.append(ap.numpy())
        dom_preds.append(dp.numpy())

    ats_preds = np.concatenate(ats_preds).squeeze(-1)
    dom_preds = np.concatenate(dom_preds)
    mae_100   = float(np.mean(np.abs(ats_preds - data['ats_scores']))) * 100.0
    dom_true  = data['domain_labels']
    dom_f1    = f1_score(dom_true, np.argmax(dom_preds, 1), average='macro', zero_division=0)
    dom_ce    = float(np.mean(
        tf.keras.losses.sparse_categorical_crossentropy(dom_true, dom_preds).numpy()
    ))
    return mae_100, dom_f1, dom_ce, dom_preds


def eval_rsg(data):
    """Returns (acc, ce)."""
    m = len(data['rsg_labels'])
    rsg_preds = []
    for s in range(0, m, BATCH_SIZE):
        e    = min(s + BATCH_SIZE, m)
        pids = data['profile_input_ids'][s:e]
        pmsk = data['profile_attention_mask'][s:e]
        _, _, rp = model([pids, pmsk, pids, pmsk], training=False)
        rsg_preds.append(rp.numpy())

    rsg_preds = np.concatenate(rsg_preds)
    rsg_true  = data['rsg_labels']
    acc = float(np.mean(np.argmax(rsg_preds, 1) == rsg_true))
    ce  = float(np.mean(
        tf.keras.losses.sparse_categorical_crossentropy(rsg_true, rsg_preds).numpy()
    ))
    return acc, ce


def composite_loss(mae_01, dom_ce, rsg_ce):
    return W_ATS * mae_01 + W_DOM * dom_ce + W_RSG * rsg_ce


print('Loss helpers and eval functions ready.')

In [ ]:
# ── Cell 11: Measure R1 baseline (before any R2 gradient) ────────────────────
print('Evaluating R1 baseline on validation set...')
r1_val_mae, r1_dom_f1, _, _ = eval_ats(ats_vl)
r1_rsg_acc, _               = eval_rsg(rsg_vl)

MAE_HARD_STOP = r1_val_mae + MAE_REGRESSION_LIMIT

print(f'  R1 val ATS MAE : {r1_val_mae:.2f}')
print(f'  R1 val Dom F1  : {r1_dom_f1:.4f}')
print(f'  R1 val RSG Acc : {r1_rsg_acc:.4f}')
print(f'  Regression hard-stop gate: val MAE > {MAE_HARD_STOP:.2f}')

In [ ]:
# ── Cell 12: Unfreeze encoder & set up differential-LR optimizers ─────────────
encoder_layer             = model.get_layer('minilm_encoder')
encoder_layer.trainable   = True
encoder_layer._bert.trainable = True

_enc_var_ids = {id(v) for v in encoder_layer.trainable_variables}

encoder_opt = tf.keras.optimizers.legacy.Adam(learning_rate=ENC_LR)
head_opt    = tf.keras.optimizers.legacy.Adam(learning_rate=HEAD_LR)

trainable = sum(int(np.prod(v.shape)) for v in model.trainable_weights)
frozen    = sum(int(np.prod(v.shape)) for v in model.non_trainable_weights)
print(f'Encoder unfrozen.')
print(f'  Trainable params : {trainable:,}')
print(f'  Frozen params    : {frozen:,}')
print(f'  Encoder opt      : Adam(lr={ENC_LR})')
print(f'  Head opt         : Adam(lr={HEAD_LR})')


def _apply_differential(grads):
    enc_gv  = [(g, v) for g, v in zip(grads, model.trainable_variables)
               if g is not None and id(v) in _enc_var_ids]
    head_gv = [(g, v) for g, v in zip(grads, model.trainable_variables)
               if g is not None and id(v) not in _enc_var_ids]
    if enc_gv:
        encoder_opt.apply_gradients(enc_gv)
    if head_gv:
        head_opt.apply_gradients(head_gv)


def train_step_ats(r_ids, r_mask, j_ids, j_mask, ats_y, dom_y):
    with tf.GradientTape() as tape:
        ats_p, dom_p, _ = model([r_ids, r_mask, j_ids, j_mask], training=True)
        l_ats = _ats_mae(ats_y, ats_p)
        l_dom = _dom_loss(dom_y, dom_p)
        loss  = W_ATS * l_ats + W_DOM * l_dom
    _apply_differential(tape.gradient(loss, model.trainable_variables))
    return float(loss), float(l_ats), float(l_dom)


def train_step_rsg(p_ids, p_mask, rsg_y):
    with tf.GradientTape() as tape:
        _, _, rsg_p = model([p_ids, p_mask, p_ids, p_mask], training=True)
        l_rsg = _rsg_loss(rsg_y, rsg_p)
        loss  = W_RSG * l_rsg
    _apply_differential(tape.gradient(loss, model.trainable_variables))
    return float(loss), float(l_rsg)


print('Train step functions ready.')

In [ ]:
# ── Cell 13: Training loop ────────────────────────────────────────────────────
print('=' * 60)
print(f'  Starting R2 training  (max={MAX_EPOCHS} epochs, patience={PATIENCE})')
print('=' * 60)

rng              = np.random.default_rng(42)
best_val_loss    = float('inf')
patience_counter = 0
best_epoch       = -1
log_rows         = []

ats_batches_per_epoch = math.ceil(n_ats_tr / BATCH_SIZE)
rsg_batches_per_epoch = math.ceil(n_rsg_tr / BATCH_SIZE)
print(f'  ATS batches/epoch: {ats_batches_per_epoch}  '
      f'RSG batches/epoch: {rsg_batches_per_epoch}\n')

for epoch in range(MAX_EPOCHS):
    ats_perm = rng.permutation(n_ats_tr)
    rsg_perm = rng.permutation(n_rsg_tr)

    ats_batch_idxs = [ats_perm[i:i + BATCH_SIZE] for i in range(0, n_ats_tr, BATCH_SIZE)]
    rsg_batch_idxs = [rsg_perm[i:i + BATCH_SIZE] for i in range(0, n_rsg_tr, BATCH_SIZE)]
    rsg_cycle      = (rsg_batch_idxs * (len(ats_batch_idxs) // len(rsg_batch_idxs) + 1))
    rsg_cycle      = rsg_cycle[:len(ats_batch_idxs)]

    epoch_ats_loss = 0.0
    epoch_rsg_loss = 0.0

    for aidx, ridx in zip(ats_batch_idxs, rsg_cycle):
        _, la, ld = train_step_ats(
            ats_tr['resume_input_ids'][aidx],
            ats_tr['resume_attention_mask'][aidx],
            ats_tr['jd_input_ids'][aidx],
            ats_tr['jd_attention_mask'][aidx],
            ats_tr['ats_scores'][aidx].astype(np.float32),
            ats_tr['domain_labels'][aidx],
        )
        epoch_ats_loss += W_ATS * la + W_DOM * ld

        _, l_rsg = train_step_rsg(
            rsg_tr['profile_input_ids'][ridx],
            rsg_tr['profile_attention_mask'][ridx],
            rsg_tr['rsg_labels'][ridx],
        )
        epoch_rsg_loss += W_RSG * l_rsg

    train_loss = (epoch_ats_loss + epoch_rsg_loss) / len(ats_batch_idxs)

    # NaN hard stop
    if np.isnan(train_loss):
        print(f'\n⛔ HARD STOP — NaN train_loss at epoch {epoch + 1}')
        break

    # Validation
    val_mae, val_dom_f1, val_dom_ce, _ = eval_ats(ats_vl)
    val_rsg_acc, val_rsg_ce            = eval_rsg(rsg_vl)
    val_loss = composite_loss(val_mae / 100.0, val_dom_ce, val_rsg_ce)

    if np.isnan(val_loss):
        print(f'\n⛔ HARD STOP — NaN val_loss at epoch {epoch + 1}')
        break

    print(
        f'Epoch {epoch+1:3d}/{MAX_EPOCHS}  '
        f'train={train_loss:.4f}  val={val_loss:.4f}  '
        f'ATS_MAE={val_mae:.2f}  Dom_F1={val_dom_f1:.4f}  RSG_Acc={val_rsg_acc:.4f}'
    )

    log_rows.append({
        'epoch':       epoch + 1,
        'train_loss':  round(train_loss, 6),
        'val_loss':    round(val_loss,   6),
        'val_ats_mae': round(val_mae,    4),
        'val_dom_f1':  round(val_dom_f1, 4),
        'val_rsg_acc': round(val_rsg_acc,4),
    })

    # Regression hard stop
    if val_mae > MAE_HARD_STOP:
        print(f'\n⛔ HARD STOP — val MAE {val_mae:.2f} > {MAE_HARD_STOP:.2f} '
              f'(R1 {r1_val_mae:.2f} + {MAE_REGRESSION_LIMIT:.1f}). Restoring best ckpt.')
        if best_epoch >= 0:
            model.load_weights(R2_CKPT_PATH)
        break

    # Checkpoint on improvement
    if val_loss < best_val_loss:
        best_val_loss    = val_loss
        best_epoch       = epoch + 1
        patience_counter = 0
        model.save_weights(R2_CKPT_PATH)
        print(f'  ✅ [Saved] best val_loss={val_loss:.4f}  ATS_MAE={val_mae:.2f} → epoch {epoch + 1}')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'\nEarly stopping (patience={PATIENCE}) at epoch {epoch + 1}')
            break

print('\nTraining loop complete.')

In [ ]:
# ── Cell 14: Save weights & training log ─────────────────────────────────────
print('Loading best checkpoint and saving r2_best_weights.h5...')
model.load_weights(R2_CKPT_PATH)
model.save_weights(R2_BEST_PATH)
print(f'  Saved: {R2_BEST_PATH}')

if log_rows:
    with open(R2_LOG_PATH, 'w', newline='') as fh:
        writer = csv.DictWriter(fh, fieldnames=log_rows[0].keys())
        writer.writeheader()
        writer.writerows(log_rows)
    print(f'  Log  : {R2_LOG_PATH}')

In [ ]:
# ── Cell 15: Final evaluation & gate report ───────────────────────────────────
val_mae_f,  val_dom_f1_f,  _, _         = eval_ats(ats_vl)
val_rsg_f,  _                           = eval_rsg(rsg_vl)
test_mae_f, test_dom_f1_f, _, test_dpred = eval_ats(ats_ts)
per_dom_f1 = f1_score(
    ats_ts['domain_labels'], np.argmax(test_dpred, 1),
    average=None, labels=list(range(7)), zero_division=0,
)

print('\n' + '=' * 60)
print('  STAGE R-2 — FINAL METRICS')
print('=' * 60)
print(f'  ATS val MAE   (0-100): {val_mae_f:.2f}   '
      f'{"✅ PASS" if val_mae_f  < GATE_VAL_MAE  else "❌ FAIL"} (gate < {GATE_VAL_MAE})')
print(f'  ATS test MAE  (0-100): {test_mae_f:.2f}   '
      f'{"✅ PASS" if test_mae_f < GATE_TEST_MAE else "❌ FAIL"} (gate < {GATE_TEST_MAE})')
print(f'  Domain val F1 (macro): {val_dom_f1_f:.4f}  '
      f'{"✅ PASS" if val_dom_f1_f > GATE_DOM_F1 else "❌ FAIL"} (gate > {GATE_DOM_F1})')
print(f'  RSG val Accuracy:      {val_rsg_f:.4f}  '
      f'{"✅ PASS" if val_rsg_f   > GATE_RSG_ACC else "❌ FAIL"} (gate > {GATE_RSG_ACC})')
print()
print('  Per-domain test F1:')
for i, (name, f1v) in enumerate(zip(DOMAIN_NAMES, per_dom_f1)):
    print(f'    [{i}] {name:12s}: {f1v:.4f}')
print()
print(f'  Best epoch     : {best_epoch}')
print(f'  Best val loss  : {best_val_loss:.4f}')
print(f'  R1 baseline MAE: {r1_val_mae:.2f}  →  delta: {val_mae_f - r1_val_mae:+.2f}')
print(f'  Output         : {R2_BEST_PATH}')
print('=' * 60)
print()
print('R2 COMPLETE — proceed to R3')

In [ ]:
# ── Cell 16: Download results to local machine ────────────────────────────────
# The weights are already saved to Google Drive (persistent).
# Use this cell to also download directly to your browser.
from google.colab import files

print('Downloading r2_best_weights.h5 ...')
files.download(R2_BEST_PATH)

print('Downloading r2_training_log.csv ...')
files.download(R2_LOG_PATH)

print('Done. Save these files to:')
print('  model/unified_model/r2_best_weights.h5')
print('  model/unified_model/r2_training_log.csv')